<a href="https://colab.research.google.com/github/Nikikanaparthi21/data266-6441/blob/main/cuda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Nikhil Kanaparthi
## DATA 266 - Homework 1
## CUDA Matrix Multiplication

In [1]:
SID4 = 6441
SEED = 6441
SLICE = 441
HP_ID = 3
CLS_A = 1
CLS_B = 7

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)

SID4: 6441
SEED: 6441
SLICE: 441
HP_ID: 3
CLS_A: 1
CLS_B: 7


### CUDA environment

The following cell verifies that a CUDA-capable GPU is available and determines
the correct architecture flag for the CUDA compiler.

In [2]:
import os
import torch

assert torch.cuda.is_available(), (
    "CUDA is unavailable. Select Runtime > Change runtime type > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
compute_capability = torch.cuda.get_device_capability(0)
sm_architecture = (
    f"sm_{compute_capability[0]}{compute_capability[1]}"
)

os.environ["SM"] = sm_architecture

print("GPU:", gpu_name)
print("Compute capability:", compute_capability)
print("NVCC architecture:", sm_architecture)

GPU: Tesla T4
Compute capability: (7, 5)
NVCC architecture: sm_75


In [3]:
!nvidia-smi
!nvcc --version

Tue Sep  1 18:18:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             12W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## CUDA matrix multiplication

The CUDA implementation divides the output matrix into two-dimensional blocks.
Each block contains `16 × 16 = 256` threads. Each thread computes one element
of the output matrix.

For output element `C[row, col]`, the thread multiplies row `row` of matrix A
by column `col` of matrix B. Shared-memory tiles reduce repeated global-memory
accesses. Boundary checks allow the same kernel to work when the matrix size
is not an exact multiple of 16.

In [4]:
%%writefile matrix_multiplication.cu

#include <cuda_runtime.h>

#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdio>
#include <cstdlib>
#include <vector>

#define CUDA_CHECK(call) do {                                      \
    cudaError_t error = (call);                                    \
    if (error != cudaSuccess) {                                    \
        std::fprintf(                                               \
            stderr,                                                 \
            "CUDA error at %s:%d: %s\n",                           \
            __FILE__,                                               \
            __LINE__,                                               \
            cudaGetErrorString(error)                               \
        );                                                          \
        std::exit(EXIT_FAILURE);                                   \
    }                                                               \
} while (0)

constexpr int TILE_SIZE = 16;


// CUDA tiled matrix-multiplication kernel
__global__ void tiledMatrixMultiply(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    __shared__ float tileA[TILE_SIZE][TILE_SIZE];
    __shared__ float tileB[TILE_SIZE][TILE_SIZE];

    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    float sum = 0.0f;

    int number_of_tiles = (N + TILE_SIZE - 1) / TILE_SIZE;

    for (int tile = 0; tile < number_of_tiles; ++tile) {
        int a_col = tile * TILE_SIZE + threadIdx.x;
        int b_row = tile * TILE_SIZE + threadIdx.y;

        if (row < N && a_col < N) {
            tileA[threadIdx.y][threadIdx.x] =
                A[row * N + a_col];
        } else {
            tileA[threadIdx.y][threadIdx.x] = 0.0f;
        }

        if (b_row < N && col < N) {
            tileB[threadIdx.y][threadIdx.x] =
                B[b_row * N + col];
        } else {
            tileB[threadIdx.y][threadIdx.x] = 0.0f;
        }

        __syncthreads();

        for (int k = 0; k < TILE_SIZE; ++k) {
            sum += (
                tileA[threadIdx.y][k] *
                tileB[k][threadIdx.x]
            );
        }

        __syncthreads();
    }

    if (row < N && col < N) {
        C[row * N + col] = sum;
    }
}


// CPU reference implementation
void cpuMatrixMultiply(
    const float* A,
    const float* B,
    float* C,
    int N
) {
    std::fill(C, C + static_cast<size_t>(N) * N, 0.0f);

    // i-k-j ordering improves CPU cache behavior.
    for (int i = 0; i < N; ++i) {
        for (int k = 0; k < N; ++k) {
            float a_value = A[i * N + k];

            for (int j = 0; j < N; ++j) {
                C[i * N + j] += a_value * B[k * N + j];
            }
        }
    }
}


double median(std::vector<double> values) {
    std::sort(values.begin(), values.end());

    size_t middle = values.size() / 2;

    if (values.size() % 2 == 1) {
        return values[middle];
    }

    return (values[middle - 1] + values[middle]) / 2.0;
}


int main(int argc, char** argv) {
    if (argc < 2) {
        std::fprintf(
            stderr,
            "Usage: %s MATRIX_SIZE [REPEATS]\n",
            argv[0]
        );
        return EXIT_FAILURE;
    }

    int N = std::atoi(argv[1]);
    int repeats = (argc >= 3) ? std::atoi(argv[2]) : 3;

    if (N <= 0 || repeats <= 0) {
        std::fprintf(
            stderr,
            "Matrix size and repetitions must be positive.\n"
        );
        return EXIT_FAILURE;
    }

    size_t element_count = static_cast<size_t>(N) * N;
    size_t bytes = element_count * sizeof(float);

    std::vector<float> A(element_count);
    std::vector<float> B(element_count);
    std::vector<float> C_cpu(element_count);
    std::vector<float> C_gpu(element_count);

    // Deterministic input values
    for (size_t i = 0; i < element_count; ++i) {
        A[i] = static_cast<float>((i * 17) % 100) / 100.0f;
        B[i] = static_cast<float>((i * 29) % 100) / 100.0f;
    }

    std::printf("Matrix size: %d x %d\n", N, N);
    std::printf("Repetitions: %d\n", repeats);
    std::printf(
        "Threads per block: %d x %d = %d\n",
        TILE_SIZE,
        TILE_SIZE,
        TILE_SIZE * TILE_SIZE
    );

    // Small CPU warm-up
    {
        constexpr int warm_size = 64;

        std::vector<float> warm_A(
            warm_size * warm_size,
            1.0f
        );

        std::vector<float> warm_B(
            warm_size * warm_size,
            1.0f
        );

        std::vector<float> warm_C(
            warm_size * warm_size
        );

        cpuMatrixMultiply(
            warm_A.data(),
            warm_B.data(),
            warm_C.data(),
            warm_size
        );
    }

    // Measure CPU baseline
    std::vector<double> cpu_times;

    for (int repeat = 0; repeat < repeats; ++repeat) {
        auto cpu_start =
            std::chrono::high_resolution_clock::now();

        cpuMatrixMultiply(
            A.data(),
            B.data(),
            C_cpu.data(),
            N
        );

        auto cpu_stop =
            std::chrono::high_resolution_clock::now();

        double cpu_ms =
            std::chrono::duration<double, std::milli>(
                cpu_stop - cpu_start
            ).count();

        cpu_times.push_back(cpu_ms);
    }

    // Allocate GPU memory
    float* d_A = nullptr;
    float* d_B = nullptr;
    float* d_C = nullptr;

    CUDA_CHECK(cudaMalloc(&d_A, bytes));
    CUDA_CHECK(cudaMalloc(&d_B, bytes));
    CUDA_CHECK(cudaMalloc(&d_C, bytes));

    dim3 threads_per_block(TILE_SIZE, TILE_SIZE);

    dim3 blocks_per_grid(
        (N + TILE_SIZE - 1) / TILE_SIZE,
        (N + TILE_SIZE - 1) / TILE_SIZE
    );

    // Full GPU warm-up
    CUDA_CHECK(
        cudaMemcpy(
            d_A,
            A.data(),
            bytes,
            cudaMemcpyHostToDevice
        )
    );

    CUDA_CHECK(
        cudaMemcpy(
            d_B,
            B.data(),
            bytes,
            cudaMemcpyHostToDevice
        )
    );

    tiledMatrixMultiply<<<
        blocks_per_grid,
        threads_per_block
    >>>(
        d_A,
        d_B,
        d_C,
        N
    );

    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaDeviceSynchronize());

    CUDA_CHECK(
        cudaMemcpy(
            C_gpu.data(),
            d_C,
            bytes,
            cudaMemcpyDeviceToHost
        )
    );

    cudaEvent_t event_start;
    cudaEvent_t event_stop;

    CUDA_CHECK(cudaEventCreate(&event_start));
    CUDA_CHECK(cudaEventCreate(&event_stop));

    std::vector<double> kernel_times;
    std::vector<double> transfer_times;
    std::vector<double> end_to_end_times;

    for (int repeat = 0; repeat < repeats; ++repeat) {
        float h2d_ms = 0.0f;
        float kernel_ms = 0.0f;
        float d2h_ms = 0.0f;

        // Measure two host-to-device transfers
        CUDA_CHECK(cudaEventRecord(event_start));

        CUDA_CHECK(
            cudaMemcpy(
                d_A,
                A.data(),
                bytes,
                cudaMemcpyHostToDevice
            )
        );

        CUDA_CHECK(
            cudaMemcpy(
                d_B,
                B.data(),
                bytes,
                cudaMemcpyHostToDevice
            )
        );

        CUDA_CHECK(cudaEventRecord(event_stop));
        CUDA_CHECK(cudaEventSynchronize(event_stop));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &h2d_ms,
                event_start,
                event_stop
            )
        );

        // Measure kernel only
        CUDA_CHECK(cudaEventRecord(event_start));

        tiledMatrixMultiply<<<
            blocks_per_grid,
            threads_per_block
        >>>(
            d_A,
            d_B,
            d_C,
            N
        );

        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaEventRecord(event_stop));
        CUDA_CHECK(cudaEventSynchronize(event_stop));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &kernel_ms,
                event_start,
                event_stop
            )
        );

        // Measure device-to-host transfer
        CUDA_CHECK(cudaEventRecord(event_start));

        CUDA_CHECK(
            cudaMemcpy(
                C_gpu.data(),
                d_C,
                bytes,
                cudaMemcpyDeviceToHost
            )
        );

        CUDA_CHECK(cudaEventRecord(event_stop));
        CUDA_CHECK(cudaEventSynchronize(event_stop));

        CUDA_CHECK(
            cudaEventElapsedTime(
                &d2h_ms,
                event_start,
                event_stop
            )
        );

        double combined_transfer_ms = h2d_ms + d2h_ms;
        double combined_end_to_end_ms =
            combined_transfer_ms + kernel_ms;

        kernel_times.push_back(kernel_ms);
        transfer_times.push_back(combined_transfer_ms);
        end_to_end_times.push_back(combined_end_to_end_ms);
    }

    // Compare CPU and GPU results
    double max_absolute_error = 0.0;
    double max_relative_error = 0.0;

    for (size_t i = 0; i < element_count; ++i) {
        double absolute_error = std::fabs(
            static_cast<double>(C_cpu[i]) -
            static_cast<double>(C_gpu[i])
        );

        double denominator = std::max(
            1.0e-6,
            std::fabs(static_cast<double>(C_cpu[i]))
        );

        double relative_error = absolute_error / denominator;

        max_absolute_error = std::max(
            max_absolute_error,
            absolute_error
        );

        max_relative_error = std::max(
            max_relative_error,
            relative_error
        );
    }

    bool correct = max_relative_error < 1.0e-3;

    double cpu_median_ms = median(cpu_times);
    double kernel_median_ms = median(kernel_times);
    double transfer_median_ms = median(transfer_times);
    double end_to_end_median_ms = median(end_to_end_times);

    double speedup = cpu_median_ms / end_to_end_median_ms;

    std::printf("\nMedian CPU time (ms): %.6f\n", cpu_median_ms);
    std::printf(
        "Median GPU kernel time (ms): %.6f\n",
        kernel_median_ms
    );
    std::printf(
        "Median H2D+D2H time (ms): %.6f\n",
        transfer_median_ms
    );
    std::printf(
        "Median GPU end-to-end time (ms): %.6f\n",
        end_to_end_median_ms
    );
    std::printf("End-to-end speedup: %.6fx\n", speedup);
    std::printf(
        "Maximum absolute error: %.8f\n",
        max_absolute_error
    );
    std::printf(
        "Maximum relative error: %.8f\n",
        max_relative_error
    );
    std::printf(
        "Correctness: %s\n",
        correct ? "PASS" : "FAIL"
    );

    // Machine-readable output for the notebook
    std::printf(
        "RESULT,%d,%.6f,%.6f,%.6f,%.6f,%.6f,%.8f,%.8f,%s\n",
        N,
        cpu_median_ms,
        kernel_median_ms,
        transfer_median_ms,
        end_to_end_median_ms,
        speedup,
        max_absolute_error,
        max_relative_error,
        correct ? "PASS" : "FAIL"
    );

    CUDA_CHECK(cudaEventDestroy(event_start));
    CUDA_CHECK(cudaEventDestroy(event_stop));

    CUDA_CHECK(cudaFree(d_A));
    CUDA_CHECK(cudaFree(d_B));
    CUDA_CHECK(cudaFree(d_C));

    return correct ? EXIT_SUCCESS : EXIT_FAILURE;
}

Writing matrix_multiplication.cu


### Build command

The source is compiled with optimization enabled and with an architecture flag
selected from the Colab GPU's compute capability.

In [5]:
!nvcc -O3 -std=c++14 -arch=$SM \
    matrix_multiplication.cu \
    -o matrix_multiplication

In [6]:
!./matrix_multiplication 256 1

Matrix size: 256 x 256
Repetitions: 1
Threads per block: 16 x 16 = 256

Median CPU time (ms): 5.137727
Median GPU kernel time (ms): 0.084864
Median H2D+D2H time (ms): 0.288672
Median GPU end-to-end time (ms): 0.373536
End-to-end speedup: 13.754302x
Maximum absolute error: 0.00001526
Maximum relative error: 0.00000024
Correctness: PASS
RESULT,256,5.137727,0.084864,0.288672,0.373536,13.754302,0.00001526,0.00000024,PASS


### Timing methodology

A small CPU operation and a full GPU kernel are executed as warm-ups before
measurement. Each required size is measured three times, and the median is
reported. GPU allocation time is excluded. GPU end-to-end time is defined as
kernel time plus host-to-device and device-to-host transfer time.

In [11]:
import pandas as pd
import subprocess
from datetime import datetime, timezone

matrix_sizes = [256, 1024, 4096]
cuda_result_rows = []

with open("RUN_LOG.txt", "a") as log_file:
    timestamp = datetime.now(timezone.utc).isoformat()

    log_file.write(
        f"\nCUDA matrix multiplication run: {timestamp}\n"
    )

    for matrix_size in matrix_sizes:
        print(f"\nRunning matrix size {matrix_size}...")

        completed = subprocess.run(
            [
                "./matrix_multiplication",
                str(matrix_size),
                "3"
            ],
            capture_output=True,
            text=True,
            check=True
        )

        print(completed.stdout)
        log_file.write(completed.stdout)
        log_file.write("\n")

        result_line = next(
            line
            for line in completed.stdout.splitlines()
            if line.startswith("RESULT,")
        )

        fields = result_line.split(",")

        cuda_result_rows.append({
            "Matrix Size": int(fields[1]),
            "CPU (ms)": float(fields[2]),
            "GPU Kernel (ms)": float(fields[3]),
            "H2D+D2H (ms)": float(fields[4]),
            "GPU End-to-End (ms)": float(fields[5]),
            "Speedup": float(fields[6]),
            "Max Absolute Error": float(fields[7]),
            "Max Relative Error": float(fields[8]),
            "Correctness": fields[9]
        })

cuda_results = pd.DataFrame(cuda_result_rows)

display(cuda_results)


Running matrix size 256...
Matrix size: 256 x 256
Repetitions: 3
Threads per block: 16 x 16 = 256

Median CPU time (ms): 3.161463
Median GPU kernel time (ms): 0.112736
Median H2D+D2H time (ms): 0.255840
Median GPU end-to-end time (ms): 0.368576
End-to-end speedup: 8.577506x
Maximum absolute error: 0.00001526
Maximum relative error: 0.00000024
Correctness: PASS
RESULT,256,3.161463,0.112736,0.255840,0.368576,8.577506,0.00001526,0.00000024,PASS


Running matrix size 1024...
Matrix size: 1024 x 1024
Repetitions: 3
Threads per block: 16 x 16 = 256

Median CPU time (ms): 219.810044
Median GPU kernel time (ms): 3.257280
Median H2D+D2H time (ms): 2.964288
Median GPU end-to-end time (ms): 6.218944
End-to-end speedup: 35.345236x
Maximum absolute error: 0.00006104
Maximum relative error: 0.00000024
Correctness: PASS
RESULT,1024,219.810044,3.257280,2.964288,6.218944,35.345236,0.00006104,0.00000024,PASS


Running matrix size 4096...
Matrix size: 4096 x 4096
Repetitions: 3
Threads per block: 16 x 1

,Matrix Size,CPU (ms),GPU Kernel (ms),H2D+D2H (ms),GPU End-to-End (ms),Speedup,Max Absolute Error,Max Relative Error,Correctness
0,256,3.161463,0.112736,0.255840,0.368576,8.577506,0.000015,2.400000e-07,PASS
1,1024,219.810044,3.257280,2.964288,6.218944,35.345236,0.000061,2.400000e-07,PASS
2,4096,26208.768718,200.267548,44.798466,245.066013,106.945750,0.001221,1.240000e-06,PASS


In [10]:
import pandas as pd

cuda_results = pd.DataFrame(cuda_result_rows)
display(cuda_results)

,Matrix Size,CPU (ms),GPU Kernel (ms),H2D+D2H (ms),GPU End-to-End (ms),Speedup,Max Absolute Error,Max Relative Error,Correctness
0,256,6.212346,0.114688,0.264800,0.380768,16.315304,0.000015,2.400000e-07,PASS
1,1024,373.067132,3.672896,2.986560,6.657856,56.034127,0.000061,2.400000e-07,PASS
2,4096,25703.000808,199.051712,46.308193,246.620766,104.220749,0.001221,1.240000e-06,PASS


In [12]:
assignment_cuda_table = cuda_results[
    [
        "Matrix Size",
        "CPU (ms)",
        "GPU Kernel (ms)",
        "H2D+D2H (ms)",
        "Speedup"
    ]
].copy()

display(
    assignment_cuda_table.round(3)
)

,Matrix Size,CPU (ms),GPU Kernel (ms),H2D+D2H (ms),Speedup
0,256,3.161,0.113,0.256,8.578
1,1024,219.810,3.257,2.964,35.345
2,4096,26208.769,200.268,44.798,106.946


### CUDA profiler availability

In [13]:
import shutil

profilers = {
    "Nsight Systems (nsys)": shutil.which("nsys"),
    "Nsight Compute (ncu)": shutil.which("ncu"),
    "nvprof": shutil.which("nvprof")
}

for profiler, path in profilers.items():
    print(f"{profiler}: {path}")

Nsight Systems (nsys): None
Nsight Compute (ncu): /usr/local/cuda/bin/ncu
nvprof: /usr/local/cuda/bin/nvprof


### CUDA profiling

The matrix-multiplication program is profiled at matrix size 1024. This size is
large enough to produce measurable kernel and transfer activity without making
the profiler run unnecessarily long.

In [14]:
!nvprof ./matrix_multiplication 1024 1 2>&1 | tee nvprof_output.txt

==6074== NVPROF is profiling process 6074, command: ./matrix_multiplication 1024 1
==6074== Profiling application: ./matrix_multiplication 1024 1
Matrix size: 1024 x 1024
Repetitions: 1
Threads per block: 16 x 16 = 256

Median CPU time (ms): 224.753055
Median GPU kernel time (ms): 5.816992
Median H2D+D2H time (ms): 3.136384
Median GPU end-to-end time (ms): 8.953376
End-to-end speedup: 25.102605x
Maximum absolute error: 0.00006104
Maximum relative error: 0.00000024
Correctness: PASS
RESULT,1024,224.753055,5.816992,3.136384,8.953376,25.102605,0.00006104,0.00000024,PASS
==6074== Profiling result:
            Type  Time(%)      Time     Calls       Avg       Min       Max  Name
 GPU activities:   72.18%  11.590ms         2  5.7949ms  5.7946ms  5.7952ms  tiledMatrixMultiply(float const *, float const *, float*, int)
                   19.99%  3.2096ms         4  802.41us  772.53us  815.18us  [CUDA memcpy HtoD]
                    7.83%  1.2576ms         2  628.80us  626.80us  630.80us  [CUD

### Profiler results and interpretation

**Profiler used:** NVIDIA `nvprof`

The profiler successfully separated kernel execution from memory transfers for
the 1024 × 1024 matrix. It reported:

- Tiled matrix-multiplication kernel: **11.590 ms total across two calls**,
  averaging **5.795 ms per call**
- Host-to-device transfers: **3.210 ms total across four calls**
- Device-to-host transfers: **1.258 ms total across two calls**
- Kernel share of reported GPU activity: **71.18%**
- Host-to-device share: **19.99%**
- Device-to-host share: **7.83%**

The profiler observed two kernel calls because the program performs one
warm-up kernel and one measured kernel. It similarly observed both warm-up and
measured memory copies. The program's CUDA-event measurement for the measured
iteration reported **5.817 ms** of kernel time and **3.136 ms** of combined
transfer time, which is consistent with the profiler's per-call results.

Profiler execution introduces additional measurement overhead, so the profiler
run is used to verify the separation of kernel and transfer activity. The
three-repeat median measurements in the main CUDA table are used for the final
performance comparison.